# 09 - RAG Retrieval with Gemini and Pinecone: Naive vs LCEL

## Learning objective

Use the vectors created in notebook **08** to compare three approaches:

```text
Implementation 0: Raw Gemini invocation without retrieval
Implementation 1: Manual or naive RAG without LCEL
Implementation 2: Declarative RAG using LCEL
```

The retrieval flow is:

```text
User question
-> Gemini query embedding, 1,536 values
-> Pinecone similarity search in namespace mediumblog1
-> Top 3 article chunks
-> Prompt containing retrieved context
-> Gemini chat model
-> Grounded answer
```

## Compatibility with the ingestion notebook

This notebook deliberately reuses the same settings:

```text
Pinecone index:            langchain-course-rag
Namespace:                 mediumblog1
Embedding model:           gemini-embedding-001
Embedding dimensionality:  1536
Similarity metric:         cosine
```

> The embedding model and dimensionality used for retrieval must match the model and dimensionality used during ingestion.


## 1. Optional package installation

The required imports already passed your command-line test. Run this cell only if a package import fails, then restart the kernel.


In [ ]:
# %pip install -U \
#     langchain-google-genai \
#     langchain-pinecone \
#     langchain-text-splitters \
#     pinecone \
#     python-dotenv \
#     truststore


## 2. Check the active Python kernel

In [1]:
import sys

print("Python executable:", sys.executable)
print("Python version:", sys.version)

Python executable: C:\Users\inkk1r04\AppData\Local\anaconda3\envs\nlp_env\python.exe
Python version: 3.10.20 | packaged by Anaconda, Inc. | (main, Mar 11 2026, 17:42:35) [MSC v.1942 64 bit (AMD64)]


## 3. Configure corporate SSL

`truststore` makes Python use the Windows certificate store. Run this before importing the API integrations.


In [2]:
import truststore

truststore.inject_into_ssl()

print("Windows trust store injected into Python SSL.")

Windows trust store injected into Python SSL.


## 4. Import the retrieval components

The notebook uses:

- `ChatGoogleGenerativeAI` for answer generation
- `GoogleGenerativeAIEmbeddings` for query embeddings
- `PineconeVectorStore` for semantic retrieval
- LCEL runnables and `StrOutputParser` for the composable chain


In [3]:
import os
from operator import itemgetter
from pathlib import Path

from dotenv import load_dotenv
from langchain_core.messages import HumanMessage
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_google_genai import (
    ChatGoogleGenerativeAI,
    GoogleGenerativeAIEmbeddings,
)
from langchain_pinecone import PineconeVectorStore
from pinecone import Pinecone


print("Imports completed successfully.")


Imports completed successfully.


## 5. Locate and load `.env`

The notebook searches for `.env` in:

1. The current notebook folder
2. The sibling `langchain-course` folder

Expected variables:

```env
GOOGLE_API_KEY=your-google-api-key
PINECONE_API_KEY=your-pinecone-api-key
INDEX_NAME=langchain-course-rag
```

Optionally, you can add:

```env
GEMINI_CHAT_MODEL=gemini-3.5-flash-lite
```

If `GEMINI_CHAT_MODEL` is not set, the notebook uses `gemini-3.5-flash-lite`.


In [4]:
CURRENT_DIR = Path.cwd().resolve()


env_candidates = [
    CURRENT_DIR / ".env",
    CURRENT_DIR.parent / "langchain-course" / ".env",
]


ENV_PATH = next(
    (
        candidate
        for candidate in env_candidates
        if candidate.exists()
    ),
    None,
)


if ENV_PATH is None:
    raise FileNotFoundError(
        "Could not find .env in the notebook folder "
        "or the sibling langchain-course folder."
    )


load_dotenv(dotenv_path=ENV_PATH)


required_variables = [
    "GOOGLE_API_KEY",
    "PINECONE_API_KEY",
    "INDEX_NAME",
]


missing_variables = [
    variable_name
    for variable_name in required_variables
    if not os.getenv(variable_name)
]


if missing_variables:
    raise ValueError(
        "Missing environment variables: "
        + ", ".join(missing_variables)
    )


print("Environment file:", ENV_PATH)

for variable_name in required_variables:
    print(
        f"{variable_name} loaded:",
        bool(os.getenv(variable_name)),
    )


Environment file: C:\Users\inkk1r04\OneDrive - Kellogg Company\Desktop\Work\0. DS Capability\Courses\Langchain\langchain-course\.env
GOOGLE_API_KEY loaded: True
PINECONE_API_KEY loaded: True
INDEX_NAME loaded: True


## 6. Define the shared RAG configuration

The namespace is essential. The ingestion notebook stored all 20 chunks inside `mediumblog1`, not the default namespace.


In [5]:
INDEX_NAME = os.environ["INDEX_NAME"]
NAMESPACE = "mediumblog1"
EMBEDDING_MODEL = "gemini-embedding-001"
EMBEDDING_DIMENSION = 1536
CHAT_MODEL = os.getenv(
    "GEMINI_CHAT_MODEL",
    "gemini-3.5-flash-lite",
)
TOP_K = 3


print("Index name:", INDEX_NAME)
print("Namespace:", NAMESPACE)
print("Embedding model:", EMBEDDING_MODEL)
print("Embedding dimension:", EMBEDDING_DIMENSION)
print("Chat model:", CHAT_MODEL)
print("Top K:", TOP_K)


Index name: langchain-course-rag
Namespace: mediumblog1
Embedding model: gemini-embedding-001
Embedding dimension: 1536
Chat model: gemini-3.5-flash-lite
Top K: 3


## 7. Connect to and validate the Pinecone index

> **This cell makes Pinecone control-plane API requests.**

It verifies that the index exists, is ready, and uses the expected 1,536 dimensions with cosine similarity.


In [6]:
pinecone_client = Pinecone(
    api_key=os.environ["PINECONE_API_KEY"]
)


existing_index_names = [
    index_info["name"]
    for index_info in pinecone_client.list_indexes()
]


if INDEX_NAME not in existing_index_names:
    raise ValueError(
        f"Pinecone index '{INDEX_NAME}' does not exist."
    )


index_description = pinecone_client.describe_index(
    INDEX_NAME
)


print("Index dimension:", index_description.dimension)
print("Index metric:", index_description.metric)
print("Index ready:", index_description.status["ready"])
print("Index host:", index_description.host)


if index_description.dimension != EMBEDDING_DIMENSION:
    raise ValueError(
        f"Pinecone dimension is {index_description.dimension}, "
        f"but retrieval is configured for {EMBEDDING_DIMENSION}."
    )


if index_description.metric != "cosine":
    raise ValueError(
        f"Expected cosine metric, found {index_description.metric}."
    )


Index dimension: 1536
Index metric: cosine
Index ready: True
Index host: langchain-course-rag-upeditj.svc.aped-4627-b74a.pinecone.io


## 8. Initialize Gemini embeddings and the Gemini chat model

Creating these client objects does **not** itself make a model request.

- The embedding client is called when the retriever embeds a question.
- The chat client is called when raw Gemini or either RAG chain generates an answer.


In [7]:
embeddings = GoogleGenerativeAIEmbeddings(
    model=EMBEDDING_MODEL,
    google_api_key=os.environ["GOOGLE_API_KEY"],
    output_dimensionality=EMBEDDING_DIMENSION,
)


llm = ChatGoogleGenerativeAI(
    model=CHAT_MODEL,
    google_api_key=os.environ["GOOGLE_API_KEY"],
    temperature=0,
    max_retries=2,
)


print("Embedding client:", type(embeddings).__name__)
print("Chat client:", type(llm).__name__)


Embedding client: GoogleGenerativeAIEmbeddings
Chat client: ChatGoogleGenerativeAI


## 9. Connect LangChain to the existing Pinecone vectors

The connection uses the index host directly and explicitly selects the `mediumblog1` namespace.


In [8]:
pinecone_index = pinecone_client.Index(
    host=index_description.host
)


vector_store = PineconeVectorStore(
    index=pinecone_index,
    embedding=embeddings,
    namespace=NAMESPACE,
)


retriever = vector_store.as_retriever(
    search_kwargs={
        "k": TOP_K,
    }
)


print("Vector store:", type(vector_store).__name__)
print("Retriever:", type(retriever).__name__)


Vector store: PineconeVectorStore
Retriever: VectorStoreRetriever


## 10. Confirm the namespace contains records

> **This cell makes one Pinecone data-plane request.**

The expected result is approximately 20 vectors in the `mediumblog1` namespace.


In [9]:
index_statistics = pinecone_index.describe_index_stats()


print("Total index vectors:", index_statistics.total_vector_count)
print("Namespaces:", index_statistics.namespaces)


namespace_statistics = index_statistics.namespaces.get(
    NAMESPACE
)


if namespace_statistics is None:
    raise ValueError(
        f"Namespace '{NAMESPACE}' was not found."
    )


print(
    f"Vectors in '{NAMESPACE}':",
    namespace_statistics.vector_count,
)


Total index vectors: 20
Namespaces: {'mediumblog1': {'vector_count': 20}}
Vectors in 'mediumblog1': 20


## 11. Define the grounded RAG prompt

The prompt instructs Gemini to answer using only the retrieved article context. It also asks Gemini to acknowledge when the retrieved chunks do not contain enough information.


In [10]:
prompt_template = ChatPromptTemplate.from_template(
    """You are a helpful RAG assistant.

Answer the question using only the supplied context.
If the context does not contain enough information, say that the answer is not available in the retrieved context.
Do not rely on outside knowledge.

Context:
{context}

Question:
{question}

Provide a clear and detailed answer:"""
)


print(prompt_template)


input_variables=['context', 'question'] input_types={} partial_variables={} messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template='You are a helpful RAG assistant.\n\nAnswer the question using only the supplied context.\nIf the context does not contain enough information, say that the answer is not available in the retrieved context.\nDo not rely on outside knowledge.\n\nContext:\n{context}\n\nQuestion:\n{question}\n\nProvide a clear and detailed answer:'), additional_kwargs={})]


In [18]:
messages_test_check = prompt_template.format_messages(
        context = "INSERTING TEST CONTEXT FOR CHECK: Pinecone is Vector Database. It uses Embeddings & Graph HNS...",
        question = "INSERTING TEST QUESTION FOR CHECK : What is Pinecone ? Explain. ",
    )

In [19]:
messages_test_check

[HumanMessage(content='You are a helpful RAG assistant.\n\nAnswer the question using only the supplied context.\nIf the context does not contain enough information, say that the answer is not available in the retrieved context.\nDo not rely on outside knowledge.\n\nContext:\nINSERTING TEST CONTEXT FOR CHECK: Pinecone is Vector Database. It uses Embeddings & Graph HNS...\n\nQuestion:\nINSERTING TEST QUESTION FOR CHECK : What is Pinecone ? Explain. \n\nProvide a clear and detailed answer:', additional_kwargs={}, response_metadata={})]

## 12. Define the document formatter

The retriever returns a list of LangChain `Document` objects. The formatter combines their text into one prompt context and labels the source chunk numbers for easier inspection.


In [11]:
def format_docs(documents):
    """Format retrieved documents into one context string."""

    formatted_documents = []

    for result_number, document in enumerate(
        documents,
        start=1,
    ):
        chunk_number = document.metadata.get(
            "chunk_number",
            "unknown",
        )

        source = document.metadata.get(
            "source",
            "unknown",
        )

        formatted_documents.append(
            f"[Retrieved result {result_number} | "
            f"chunk {chunk_number} | source: {source}]"
            f"{document.page_content}"
        )

    return "".join(formatted_documents)


## 13. Inspect retrieval before involving the LLM

This is the most important diagnostic step in RAG.

> **This cell makes one Gemini embedding request and one Pinecone query request. It does not call the chat model.**

If the answer-generating model performs poorly, first check whether the correct chunks were retrieved.


In [12]:
query = "What is Pinecone in machine learning?"


retrieved_documents = retriever.invoke(query)


print("Query:", query)
print("Retrieved documents:", len(retrieved_documents))


for result_number, document in enumerate(
    retrieved_documents,
    start=1,
):
    print("=" * 80)
    print("Result:", result_number)
    print(
        "Chunk number:",
        document.metadata.get("chunk_number"),
    )
    print(
        "Source:",
        document.metadata.get("source"),
    )
    print("Content:")
    print(document.page_content)
    print()


Query: What is Pinecone in machine learning?
Retrieved documents: 3
Result: 1
Chunk number: 8.0
Source: https://medium.com/@EjiroOnose/vector-database-what-is-it-and-why-you-should-know-it-ae7e7dca82a4
Content:
Pinecone is designed to be fast and scalable, allowing for efficient retrieval of similar data points based on their vector representations.
It can handle large-scale ML applications with millions or billions of data points.
Pinecone provides infrastructure management or maintenance to its users.
Pinecone can handle high query throughput and low latency search.
Pinecone is a secure platform that meets the security needs of businesses and organizations.
Pinecone is designed to be user-friendly and accessible via its simple API for storing and retrieving vector data, making it easy to integrate into existing ML workflows.
Pinecone supports real-time updates, allowing for efficient updates to the vector database as new data points are added. This ensures that the vector database re

In [47]:
retrieved_documents[0]

Document(id='82b3b78c4ae7e2f828465220465d8192c4e89039b0c221834b4b88a6b2d6bfeb', metadata={'chunk_number': 8.0, 'embedding_dimension': 1536.0, 'embedding_model': 'gemini-embedding-001', 'source': 'https://medium.com/@EjiroOnose/vector-database-what-is-it-and-why-you-should-know-it-ae7e7dca82a4', 'source_file': 'mediumblog1.txt'}, page_content='Pinecone is designed to be fast and scalable, allowing for efficient retrieval of similar data points based on their vector representations.\nIt can handle large-scale ML applications with millions or billions of data points.\nPinecone provides infrastructure management or maintenance to its users.\nPinecone can handle high query throughput and low latency search.\nPinecone is a secure platform that meets the security needs of businesses and organizations.\nPinecone is designed to be user-friendly and accessible via its simple API for storing and retrieving vector data, making it easy to integrate into existing ML workflows.\nPinecone supports rea

In [15]:
formatted_document = format_docs(retrieved_documents)

In [16]:
formatted_document

'[Retrieved result 1 | chunk 8.0 | source: https://medium.com/@EjiroOnose/vector-database-what-is-it-and-why-you-should-know-it-ae7e7dca82a4]Pinecone is designed to be fast and scalable, allowing for efficient retrieval of similar data points based on their vector representations.\nIt can handle large-scale ML applications with millions or billions of data points.\nPinecone provides infrastructure management or maintenance to its users.\nPinecone can handle high query throughput and low latency search.\nPinecone is a secure platform that meets the security needs of businesses and organizations.\nPinecone is designed to be user-friendly and accessible via its simple API for storing and retrieving vector data, making it easy to integrate into existing ML workflows.\nPinecone supports real-time updates, allowing for efficient updates to the vector database as new data points are added. This ensures that the vector database remains up-to-date and accurate over time.\nPinecone can be synced

In [17]:
type(formatted_document)

str

## 14. Optional: inspect Pinecone similarity scores

The standard retriever returns documents but not their numeric scores. This direct vector-store call returns both.

> **This cell makes another Gemini embedding request and Pinecone query request. Skip it if you want to minimize API usage.**


In [48]:
scored_results = vector_store.similarity_search_with_score(
    query=query,
    k=TOP_K,
)


for result_number, (document, score) in enumerate(
    scored_results,
    start=1,
):
    print("=" * 80)
    print("Result:", result_number)
    print("Cosine similarity score:", round(score, 4))
    print("Chunk:", document.metadata.get("chunk_number"))
    print(document.page_content[:800])
    print()


Result: 1
Cosine similarity score: 0.7668
Chunk: 8.0
Pinecone is designed to be fast and scalable, allowing for efficient retrieval of similar data points based on their vector representations.
It can handle large-scale ML applications with millions or billions of data points.
Pinecone provides infrastructure management or maintenance to its users.
Pinecone can handle high query throughput and low latency search.
Pinecone is a secure platform that meets the security needs of businesses and organizations.
Pinecone is designed to be user-friendly and accessible via its simple API for storing and retrieving vector data, making it easy to integrate into existing ML workflows.
Pinecone supports real-time updates, allowing for efficient updates to the vector database as new data points are added. This ensures that the vector database remains up-to

Result: 2
Cosine similarity score: 0.7567
Chunk: 7.0
Weaviate can store and search vectors from various data modalities, including images, text, 

In [58]:
scored_results[0][1], scored_results[1][1],  scored_results[2][1]

(0.766783535, 0.756746829, 0.700125754)

# Implementation 0: Raw Gemini without RAG

This is the baseline comparison.

```text
Question -> Gemini -> Answer
```

No Pinecone retrieval occurs. Gemini answers using its pretrained knowledge.

> **This cell makes one Gemini chat-model request.**


In [13]:
raw_response = llm.invoke([
    HumanMessage(content=query)
])


print("=" * 80)
print("IMPLEMENTATION 0: RAW GEMINI, NO RAG")
print("=" * 80)
print("Question:", query)
print("Answer:")
print(raw_response.content)


C:\Users\inkk1r04\AppData\Local\anaconda3\envs\nlp_env\lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


IMPLEMENTATION 0: RAW GEMINI, NO RAG
Question: What is Pinecone in machine learning?
Answer:
[{'type': 'text', 'text': 'In machine learning and artificial intelligence, **Pinecone** is a fully managed, cloud-native **vector database**. \n\nTo understand why Pinecone is important, you first need to understand the concept of **vector embeddings** in modern AI.\n\n---\n\n### The Background: Vector Embeddings\nIn machine learning, data like text, images, audio, and video are converted into long arrays of numbers called **vectors** (or embeddings) by models like OpenAI’s GPT, BERT, or CLIP. \n* These numbers capture the **semantic meaning** of the data. \n* Concepts that are similar are placed close to each other in a multi-dimensional mathematical space. \n\nFor example, the vector for "king" minus "man" plus "woman" will land very close to the vector for "queen."\n\n### What Does Pinecone Do?\nOnce you have millions or billions of these vectors, you need a way to search through them insta

# Implementation 1: Naive RAG without LCEL

This function executes every step manually:

```text
1. Retrieve the top 3 chunks
2. Format the chunks into a context string
3. Insert context and question into the prompt
4. Invoke Gemini
5. Return the answer text
```

This approach is excellent for understanding the core RAG mechanics because every intermediate object is visible.


In [20]:
def retrieval_chain_without_lcel(query_text: str):
    """Run a manual RAG pipeline without LCEL."""

    # Step 1: Embed the query and retrieve relevant chunks.
    documents = retriever.invoke(query_text)

    # Step 2: Convert the Document objects into prompt context.
    context = format_docs(documents)

    # Step 3: Insert the context and question into the prompt.
    messages = prompt_template.format_messages(
        context=context,
        question=query_text,
    )

    # Step 4: Generate an answer with Gemini.
    response = llm.invoke(messages)

    # Step 5: Return both answer and retrieved documents.
    return {
        "answer": response.content,
        "documents": documents,
        "formatted_context": context,
    }


## 15. Run the naive RAG implementation

> **This cell makes one Gemini embedding request, one Pinecone query request, and one Gemini chat-model request.**


In [21]:
naive_result = retrieval_chain_without_lcel(
    query
)


print("=" * 80)
print("IMPLEMENTATION 1: NAIVE RAG WITHOUT LCEL")
print("=" * 80)
print("Question:", query)
print("Answer:")
print(naive_result["answer"])
print("Retrieved chunk numbers:")
print([
    document.metadata.get("chunk_number")
    for document in naive_result["documents"]
])


C:\Users\inkk1r04\AppData\Local\anaconda3\envs\nlp_env\lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


IMPLEMENTATION 1: NAIVE RAG WITHOUT LCEL
Question: What is Pinecone in machine learning?
Answer:
[{'type': 'text', 'text': 'Based on the provided context, Pinecone is described as a fully managed cloud-based vector database designed to make it easy for businesses and organizations to build and deploy large-scale machine learning (ML) applications with millions or billions of data points. \n\nIts features and capabilities include:\n* **Performance:** It is designed to be fast, scalable, capable of handling high query throughput, and provides low-latency search for efficiently retrieving similar data points based on their vector representations.\n* **Ease of Use:** It is user-friendly and accessible via a simple API for storing and retrieving vector data, making it easy to integrate into existing ML workflows.\n* **Updates and Synchronization:** It supports real-time updates as new data points are added to keep the database accurate over time. It can also be synced with data from various

## 16. Inspect the exact prompt context used by naive RAG

This cell makes no API calls. It shows what was supplied to Gemini as grounding context.


In [22]:
print(naive_result["formatted_context"])

[Retrieved result 1 | chunk 8.0 | source: https://medium.com/@EjiroOnose/vector-database-what-is-it-and-why-you-should-know-it-ae7e7dca82a4]Pinecone is designed to be fast and scalable, allowing for efficient retrieval of similar data points based on their vector representations.
It can handle large-scale ML applications with millions or billions of data points.
Pinecone provides infrastructure management or maintenance to its users.
Pinecone can handle high query throughput and low latency search.
Pinecone is a secure platform that meets the security needs of businesses and organizations.
Pinecone is designed to be user-friendly and accessible via its simple API for storing and retrieving vector data, making it easy to integrate into existing ML workflows.
Pinecone supports real-time updates, allowing for efficient updates to the vector database as new data points are added. This ensures that the vector database remains up-to-date and accurate over time.
Pinecone can be synced with da

# Implementation 2: RAG with LCEL

LCEL expresses the same flow declaratively with the pipe operator:

```text
question
   ├── remains available as question
   └── retriever -> format_docs -> context

{question, context}
-> prompt
-> Gemini
-> string output parser
```

The LCEL chain is more concise and supports standard runnable features such as `invoke`, `stream`, `ainvoke`, and `batch`.


In [23]:
def create_retrieval_chain_with_lcel():
    """
    Create a composable LCEL RAG chain.

    Flow:
    1. itemgetter("question") extracts the question string from the input dictionary.
       itemgetter("my_question") approximately means: lambda x: x["my_question"]
       
    2. The retriever uses the Gemini embedding model to embed the question and 
       retrieves the top N matching chunksfrom Pinecone.
       
    3. format_docs combines only the retrieved chunks into one context string.
       The context string does not contain the original question.
       
    4. RunnablePassthrough.assign() preserves the original question and adds the 
       context string, producing: {"question": ..., "context": ...}
       
    5. prompt_template receives that dictionary and creates formatted chat messages.
    
    6. The LLM receives the formatted messages and generates an AIMessage.
    
    7. StrOutputParser() extracts the textual content from the AIMessage and 
       returns a normal Python string.
    """

    retrieval_chain = (
        RunnablePassthrough.assign(
            context=(
                itemgetter("question")
                | retriever
                | format_docs
            )
        )
        | prompt_template
        | llm
        | StrOutputParser()
    )

    return retrieval_chain


chain_with_lcel = create_retrieval_chain_with_lcel()


print("LCEL chain created:")
print(chain_with_lcel)

LCEL chain created:
first=RunnableAssign(mapper={
  context: RunnableLambda(itemgetter('question'))
           | VectorStoreRetriever(tags=['PineconeVectorStore', 'GoogleGenerativeAIEmbeddings'], vectorstore=<langchain_pinecone.vectorstores.PineconeVectorStore object at 0x000001EEFEBC35E0>, search_kwargs={'k': 3})
           | RunnableLambda(format_docs)
}) middle=[ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template='You are a helpful RAG assistant.\n\nAnswer the question using only the supplied context.\nIf the context does not contain enough information, say that the answer is not available in the retrieved context.\nDo not rely on outside knowledge.\n\nContext:\n{context}\n\nQuestion:\n{question}\n\nProvide a clear and detailed answer:'), additional_kwargs={})]), ChatGoogleGenerativeAI(metadat

### Not Required to Run. Just for Understanding Runnable Passthrough

> ** Below cells showacases 3 test examples **

In [41]:
itemgetter("question")

operator.itemgetter('question')

In [42]:
# Initialize a chain that adds a 'computed_key' while keeping 'input_key'
test_chain = RunnablePassthrough.assign(
    computed_key=lambda x: x["input_key"] * 2
)

test_result = test_chain.invoke({"input_key": 5})
print(test_result)  

{'input_key': 5, 'computed_key': 10}


In [45]:
test_chain_2 = RunnablePassthrough.assign(
                my_context=(
                    itemgetter("my_question") #  itemgetter("my_question") approximately means: lambda x: x["my_question"]
                )
        )

test_result_2 = test_chain_2.invoke({"my_question": "Hey What Is Pinecone ?"})
print(test_result_2)  

{'my_question': 'Hey What Is Pinecone ?', 'my_context': 'Hey What Is Pinecone ?'}


In [44]:
from langchain_core.runnables import RunnableLambda

# Define a mock retriever or format function for the example
def mock_format_1(text): return f"Context 1 from {text.upper()}"
def mock_format_2(text): return f"Context 2 from {text.lower()}"

# 1. Chain setup with multiple assigned keys
test_chain_3 = RunnablePassthrough.assign(
    my_context_1 = (itemgetter("my_question") | RunnableLambda(mock_format_1)),
    my_context_2 = (itemgetter("my_question") | RunnableLambda(mock_format_2)),
    my_context_3 = lambda x: f"Static or lambda context for {x['my_question']}"
)

# 2. Invoke the chain
test_result_3 = test_chain_3.invoke({"my_question": "Hey What Is Pinecone ?"})
print(test_result_3)

{'my_question': 'Hey What Is Pinecone ?', 'my_context_1': 'Context 1 from HEY WHAT IS PINECONE ?', 'my_context_2': 'Context 2 from hey what is pinecone ?', 'my_context_3': 'Static or lambda context for Hey What Is Pinecone ?'}


## 17. Run the LCEL RAG implementation

> **This cell makes one Gemini embedding request, one Pinecone query request, and one Gemini chat-model request.**


In [25]:
lcel_result = chain_with_lcel.invoke({
    "question": query,
})


print("=" * 80)
print("IMPLEMENTATION 2: RAG WITH LCEL")
print("=" * 80)
print("Question:", query)
print("Answer:")
print(lcel_result)


C:\Users\inkk1r04\AppData\Local\anaconda3\envs\nlp_env\lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


IMPLEMENTATION 2: RAG WITH LCEL
Question: What is Pinecone in machine learning?
Answer:
Based on the provided context, Pinecone is described as:

* A fully managed cloud-based vector database designed to make it easy for businesses and organizations to build and deploy large-scale ML (machine learning) applications.
* Designed to be fast and scalable, allowing for the efficient retrieval of similar data points based on their vector representations.
* Capable of handling large-scale ML applications with millions or billions of data points.
* Designed to be user-friendly and accessible via a simple API for storing and retrieving vector data, making it easy to integrate into existing ML workflows.


## 18. Compare the three outputs

This cell makes no additional API calls.

The intended comparison is:

```text
Raw Gemini
-> May provide a broad answer from pretrained knowledge

Naive RAG
-> Uses the article chunks and exposes every manual step

LCEL RAG
-> Uses the same retrieval architecture in a composable pipeline
```


In [27]:
print("=" * 80)
print("RAW GEMINI, NO RAG")
print("=" * 80)
print(raw_response.content)


print("" + "=" * 80)
print("NAIVE RAG WITHOUT LCEL")
print("=" * 80)
print(naive_result["answer"])


print("" + "=" * 80)
print("RAG WITH LCEL")
print("=" * 80)
print(lcel_result)


RAW GEMINI, NO RAG
[{'type': 'text', 'text': 'In machine learning and artificial intelligence, **Pinecone** is a fully managed, cloud-native **vector database**. \n\nTo understand why Pinecone is important, you first need to understand the concept of **vector embeddings** in modern AI.\n\n---\n\n### The Background: Vector Embeddings\nIn machine learning, data like text, images, audio, and video are converted into long arrays of numbers called **vectors** (or embeddings) by models like OpenAI’s GPT, BERT, or CLIP. \n* These numbers capture the **semantic meaning** of the data. \n* Concepts that are similar are placed close to each other in a multi-dimensional mathematical space. \n\nFor example, the vector for "king" minus "man" plus "woman" will land very close to the vector for "queen."\n\n### What Does Pinecone Do?\nOnce you have millions or billions of these vectors, you need a way to search through them instantly based on mathematical similarity (e.g., *"Find the 5 documents most r

## 19. Optional LCEL streaming demonstration

LCEL supports streaming through the same chain interface.

> **This cell performs a fresh retrieval and generation run, so it makes additional Gemini and Pinecone requests. Leave it commented unless you specifically want to test streaming.**


In [ ]:
# for output_chunk in chain_with_lcel.stream({
#     "question": query,
# }):
#     print(
#         output_chunk,
#         end="",
#         flush=True,
#     )


## 20. Optional LCEL batch demonstration

Batching is useful when processing multiple questions with the same pipeline.

> **This cell performs multiple retrieval and generation runs. Leave it commented to conserve API usage.**


In [ ]:
# questions = [
#     {"question": "What is a vector database?"},
#     {"question": "Why are embeddings useful?"},
#     {"question": "What is Pinecone used for?"},
# ]
#
#
# batch_results = chain_with_lcel.batch(
#     questions
# )
#
#
# for question_input, answer in zip(
#     questions,
#     batch_results,
# ):
#     print("=" * 80)
#     print("Question:", question_input["question"])
#     print("Answer:", answer)
#     print()


## 21. Approximate API calls made by the active demonstration cells

If every main demonstration cell is run once:

```text
Retrieval inspection:
1 Gemini embedding request + 1 Pinecone query

Raw Gemini:
1 Gemini chat request

Naive RAG:
1 Gemini embedding request + 1 Pinecone query + 1 Gemini chat request

LCEL RAG:
1 Gemini embedding request + 1 Pinecone query + 1 Gemini chat request
```

Approximate total:

```text
Gemini embedding requests: 3
Gemini chat requests:      3
Pinecone semantic queries: 3
```

The optional scored search, streaming, and batch cells generate additional requests.


## 22. Key takeaway

The naive and LCEL implementations perform the same conceptual RAG work:

```text
retrieve -> format -> prompt -> generate
```

The difference is orchestration style:

```text
Naive RAG
-> Explicit Python statements
-> Best for learning and debugging each step

LCEL RAG
-> Declarative runnable composition
-> Easier streaming, async, batching, reuse, and composition
```

## Important caution

RAG quality depends first on retrieval quality. Before changing prompts or the chat model, inspect:

- Retrieved chunk numbers
- Retrieved text
- Similarity scores
- Chunking boundaries
- Whether the article actually contains the answer
